# Arms Race

**Analytical purpose:** How did the nuclear arms race evolve alongside the Olympic rivalry?

This notebook is the official chart-specific preprocessing pipeline. Shared Olympic/geography logic lives in `common.py`.

In [ ]:
from pathlib import Path
import sys

CHARTS_DIR = Path.cwd()
if CHARTS_DIR.name != 'charts':
    candidates = [p / 'preprocessing' / 'charts' for p in [Path.cwd(), *Path.cwd().parents]]
    CHARTS_DIR = next((p for p in candidates if (p / 'common.py').exists()), None)
    if CHARTS_DIR is None:
        raise RuntimeError('Run this notebook from the repository or preprocessing/charts directory.')
sys.path.insert(0, str(CHARTS_DIR))
from common import *
ensure_output_dirs()

In [ ]:
import pandas as pd
import numpy as np

common = load_common()
nuclear = pd.read_csv(NUCLEAR_SOURCE)

pivot = (
    nuclear[nuclear["Year"].between(1945, 1991)]
    .pivot(index="Year", columns="Entity", values="Number of nuclear warheads")
    .reset_index()
    .rename(columns={"United States": "USA_Warheads", "Russia": "USSR_Warheads"})
)

usa = common[(common.NOC == "USA") & (common.ParticipationStatus == "participated")].set_index("Year")
urs = common[(common.NOC == "URS") & (common.ParticipationStatus == "participated")].set_index("Year")
city = {year: common.loc[common.Year.eq(year), "City"].dropna().iloc[0] for year in RIVALRY_YEARS}

rows = []
for r in pivot.itertuples(index=False):
    year = int(r.Year)
    rows.append({
        "Year": year,
        "USA_Warheads": int(r.USA_Warheads),
        "USSR_Warheads": int(r.USSR_Warheads),
        "USSR_SourceEntity": "Russia",
        "IsOlympicYear": year in RIVALRY_YEARS,
        "City": city.get(year, ""),
        "USA_TotalMedals": int(usa.loc[year, "TotalMedals"]) if year in usa.index else np.nan,
        "USSR_TotalMedals": int(urs.loc[year, "TotalMedals"]) if year in urs.index else np.nan,
        "USA_GoldMedals": int(usa.loc[year, "GoldMedals"]) if year in usa.index else np.nan,
        "USSR_GoldMedals": int(urs.loc[year, "GoldMedals"]) if year in urs.index else np.nan,
        "BoycottBy": BOYCOTTS.get(year, ""),
    })

out = pd.DataFrame(rows)
assert set(out.Year) == set(range(1945, 1992))
path = FINAL_DIR / "arms_race.csv"
out.to_csv(path, index=False)
print(f"Wrote {path.relative_to(REPO_ROOT)}: {len(out)} rows")
out.head()